# CAFA-6 Ensemble Inference For New Protein Sequences

Notebook này dùng model ensemble đã train để infer GO terms cho sequence mới bất kỳ.

## Input cần có

Một folder artifact đã lưu từ training:

```text
cafa6_high_performance_artifacts/
  config.json
  go_terms.json
  go_metadata.json
  branch_checkpoints/
    esm_mlp.pt
    protcnn.pt
    bilstm_attention.pt
  cafa6_high_performance_models.pt   # optional nếu có branch_checkpoints
```

Nếu dùng nhánh `esm_mlp`, môi trường cũng cần tải được pretrained ESM model:

```text
facebook/esm2_t30_150M_UR50D
```

Trên Kaggle, nếu không có internet, hãy add Hugging Face model cache/dataset phù hợp hoặc chạy notebook trong môi trường đã cache model.

## Cách nhập sequence

Notebook hỗ trợ 3 cách:

1. Điền thủ công vào `MANUAL_RECORDS`.
2. Đọc FASTA từ `INPUT_FASTA`.
3. Đọc CSV/TSV có cột protein id và sequence từ `INPUT_TABLE`.

Output gồm bảng:

```text
ProteinID | GO_Term | Score | Aspect | Name
```

In [ ]:
# =============================
# 0. Configuration
# =============================
from pathlib import Path
import json
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[CONFIG] DEVICE={DEVICE}")

# Artifact discovery. Set ARTIFACT_DIR manually if auto-discovery fails.
ARTIFACT_CANDIDATES = [
    Path("/kaggle/input/cafa6-high-performance-artifacts/cafa6_high_performance_artifacts"),
    Path("/kaggle/input/cafa6-offline-realtime-solution/cafa6_high_performance_artifacts"),
    Path("/kaggle/input/notebooks/qundngdngminhqun/cafa6/cafa6_high_performance_artifacts"),
    Path("/kaggle/working/cafa6_high_performance_artifacts"),
    Path("cafa6_high_performance_artifacts"),
]

# Manual input examples. Replace these with your sequences.
MANUAL_RECORDS = [
    ("example_protein_1", "MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQV"),
]

# Optional file inputs. Leave as None if not used.
INPUT_FASTA = None  # e.g. "/kaggle/input/my-proteins/query.fasta"
INPUT_TABLE = None  # e.g. "/kaggle/input/my-proteins/query.csv" or .tsv
TABLE_ID_COL = "protein_id"
TABLE_SEQUENCE_COL = "sequence"

# Inference controls.
TOP_K = 100
THRESHOLD = None  # None means use best threshold stored in config.json.
BATCH_SIZE = None # None means use config embedding_batch_size.

OUTPUT_DIR = Path("/kaggle/working/cafa6_new_sequence_predictions") if Path("/kaggle/working").exists() else Path("cafa6_new_sequence_predictions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def find_artifact_dir():
    for p in ARTIFACT_CANDIDATES:
        if (p / "config.json").exists() and ((p / "branch_checkpoints").exists() or (p / "cafa6_high_performance_models.pt").exists()):
            return p
    if Path("/kaggle/input").exists():
        for p in Path("/kaggle/input").rglob("cafa6_high_performance_artifacts"):
            if (p / "config.json").exists():
                return p
    raise FileNotFoundError("Cannot find cafa6_high_performance_artifacts. Attach it as input or set ARTIFACT_DIR manually.")

ARTIFACT_DIR = find_artifact_dir()
print(f"[PATH] ARTIFACT_DIR={ARTIFACT_DIR}")
print(f"[PATH] OUTPUT_DIR={OUTPUT_DIR}")

In [ ]:
# =============================
# 1. Load input sequences
# =============================
def clean_sequence(seq):
    seq = str(seq).strip().upper()
    seq = re.sub(r"\s+", "", seq)
    return seq


def extract_fasta_id(header_line):
    token = header_line.strip()
    if token.startswith(">"):
        token = token[1:]
    token = token.split()[0]
    if "|" in token:
        parts = token.split("|")
        if len(parts) >= 2 and parts[1]:
            return parts[1]
    return token


def load_fasta(path):
    records = []
    current_id = None
    current_seq = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current_id is not None:
                    records.append((current_id, clean_sequence("".join(current_seq))))
                current_id = extract_fasta_id(line)
                current_seq = []
            else:
                current_seq.append(line)
        if current_id is not None:
            records.append((current_id, clean_sequence("".join(current_seq))))
    return records


def load_table(path, id_col, seq_col):
    path = Path(path)
    sep = "\t" if path.suffix.lower() in {".tsv", ".tab"} else ","
    df = pd.read_csv(path, sep=sep)
    if id_col not in df.columns or seq_col not in df.columns:
        raise ValueError(f"Table must contain columns {id_col!r} and {seq_col!r}. Found: {df.columns.tolist()}")
    return [(str(r[id_col]), clean_sequence(r[seq_col])) for _, r in df.iterrows()]

records = []
if INPUT_FASTA:
    records.extend(load_fasta(INPUT_FASTA))
if INPUT_TABLE:
    records.extend(load_table(INPUT_TABLE, TABLE_ID_COL, TABLE_SEQUENCE_COL))
if MANUAL_RECORDS:
    records.extend([(str(pid), clean_sequence(seq)) for pid, seq in MANUAL_RECORDS])

# Drop empty sequences and duplicate IDs while preserving order.
seen = set()
clean_records = []
for pid, seq in records:
    if not seq:
        continue
    base_id = pid
    if pid in seen:
        suffix = 2
        while f"{base_id}_{suffix}" in seen:
            suffix += 1
        pid = f"{base_id}_{suffix}"
    seen.add(pid)
    clean_records.append((pid, seq))
records = clean_records

if not records:
    raise ValueError("No input sequences found. Fill MANUAL_RECORDS or set INPUT_FASTA/INPUT_TABLE.")

print(f"[INPUT] records={len(records):,}")
display(pd.DataFrame([{"ProteinID": pid, "Length": len(seq), "SequencePreview": seq[:60]} for pid, seq in records]).head(20))

In [ ]:
# =============================
# 2. Model definitions and artifact loading
# =============================
AA = 'ACDEFGHIKLMNPQRSTVWY'
aa_to_idx = {aa: i + 1 for i, aa in enumerate(AA)}

class EmbeddingMLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dims=(1024, 512), dropout=0.35):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

class ProtCNN(nn.Module):
    def __init__(self, output_dim, vocab_size=21, emb_dim=128, dropout=0.35):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.conv3 = nn.Sequential(nn.Conv1d(emb_dim, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU())
        self.conv5 = nn.Sequential(nn.Conv1d(emb_dim, 256, 5, padding=2), nn.BatchNorm1d(256), nn.ReLU())
        self.conv7 = nn.Sequential(nn.Conv1d(emb_dim, 256, 7, padding=3), nn.BatchNorm1d(256), nn.ReLU())
        self.conv = nn.Sequential(nn.Conv1d(768, 512, 3, padding=1), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout))
        self.head = nn.Sequential(nn.Linear(1024, 1024), nn.ReLU(), nn.Dropout(dropout), nn.Linear(1024, output_dim))
    def forward(self, x):
        x = self.embedding(x).transpose(1, 2)
        x = torch.cat([self.conv3(x), self.conv5(x), self.conv7(x)], dim=1)
        x = self.conv(x)
        gap = torch.mean(x, dim=2)
        gmp = torch.max(x, dim=2).values
        return self.head(torch.cat([gap, gmp], dim=1))

class BiLSTMAttention(nn.Module):
    def __init__(self, output_dim, vocab_size=21, emb_dim=128, hidden=256, dropout=0.35):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm1 = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.attn = nn.MultiheadAttention(hidden * 2, num_heads=8, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(hidden * 2)
        self.lstm2 = nn.LSTM(hidden * 2, hidden // 2, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(hidden * 2, 512), nn.ReLU(), nn.Dropout(dropout), nn.Linear(512, output_dim))
    def forward(self, x):
        pad_mask = x.eq(0)
        x = self.embedding(x)
        x, _ = self.lstm1(x)
        attn_out, _ = self.attn(x, x, x, key_padding_mask=pad_mask)
        x = self.norm(x + self.dropout(attn_out))
        x, _ = self.lstm2(x)
        valid = (~pad_mask).unsqueeze(-1).to(x.dtype)
        gap = (x * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1.0)
        masked = x.masked_fill(pad_mask.unsqueeze(-1), -1e4)
        gmp = masked.max(dim=1).values
        return self.head(torch.cat([gap, gmp], dim=1))

cfg = json.loads((ARTIFACT_DIR / "config.json").read_text())
selected_terms = json.loads((ARTIFACT_DIR / "go_terms.json").read_text())
go_meta = json.loads((ARTIFACT_DIR / "go_metadata.json").read_text())
output_dim = len(selected_terms)


def load_branch_states():
    states = {}
    ckpt_dir = ARTIFACT_DIR / "branch_checkpoints"
    if ckpt_dir.exists():
        for p in sorted(ckpt_dir.glob("*.pt")):
            obj = torch.load(p, map_location="cpu")
            branch = obj.get("branch", p.stem)
            states[branch] = obj.get("state_dict", obj)
            print(f"[CKPT] loaded branch {branch}: {p.name}")
    full_ckpt = ARTIFACT_DIR / "cafa6_high_performance_models.pt"
    if full_ckpt.exists():
        obj = torch.load(full_ckpt, map_location="cpu")
        for branch, state in obj.get("model_states", {}).items():
            states.setdefault(branch, state)
    return states

model_states = load_branch_states()
print("[CKPT] branches:", sorted(model_states))
print("[CONFIG] threshold:", cfg.get("best_threshold_micro_f1"))
print("[CONFIG] active weights:", cfg.get("active_ensemble_weights"))

In [ ]:
# =============================
# 3. ESM embedding for new sequences
# =============================
def prepare_lm_sequences(batch_seqs, model_name):
    if 'prot_bert' in model_name.lower() or 'protbert' in model_name.lower():
        cleaned = [re.sub(r"[UZOB]", "X", seq.upper()) for seq in batch_seqs]
        return [' '.join(list(seq)) for seq in cleaned]
    return [seq.upper() for seq in batch_seqs]


def pool_lm_outputs(outputs, attention_mask, pooling='mean'):
    hidden = outputs.last_hidden_state
    mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
    if pooling == 'cls':
        return hidden[:, 0]
    return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

@torch.no_grad()
def extract_esm_embeddings_for_records(records):
    if 'esm_mlp' not in model_states:
        return None
    from transformers import AutoTokenizer, AutoModel
    model_name = cfg['embedding_model_name']
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()
    if DEVICE == 'cuda' and cfg.get('use_fp16_embeddings', True):
        model.half()
    batch_size = BATCH_SIZE or int(cfg.get('embedding_batch_size', 8))
    vectors = []
    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]
        seqs = [seq[:cfg['embedding_max_length']] for _, seq in batch]
        seqs = prepare_lm_sequences(seqs, model_name)
        toks = tokenizer(seqs, return_tensors='pt', padding=True, truncation=True, max_length=cfg['embedding_max_length'])
        toks = {k: v.to(DEVICE) for k, v in toks.items()}
        if DEVICE == 'cuda':
            with torch.cuda.amp.autocast(enabled=cfg.get('use_fp16_embeddings', True)):
                pooled = pool_lm_outputs(model(**toks), toks['attention_mask'], cfg.get('embedding_pooling', 'mean'))
        else:
            pooled = pool_lm_outputs(model(**toks), toks['attention_mask'], cfg.get('embedding_pooling', 'mean'))
        vectors.append(pooled.float().cpu().numpy())
        print(f"[ESM] embedded {min(start + batch_size, len(records)):,}/{len(records):,}")
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return np.vstack(vectors).astype(np.float32)

Xesm = extract_esm_embeddings_for_records(records)
if Xesm is not None:
    print("[ESM] Xesm", Xesm.shape)

In [ ]:
# =============================
# 4. Sequence tensor for ProtCNN/BiLSTM
# =============================
def encode_sequence(seq, max_len):
    arr = np.zeros(max_len, dtype=np.int64)
    for i, aa in enumerate(seq[:max_len]):
        arr[i] = aa_to_idx.get(aa, 0)
    return arr

need_sequence_tensor = any(k in model_states for k in ['protcnn', 'bilstm_attention'])
if need_sequence_tensor:
    Xseq = np.vstack([encode_sequence(seq, cfg['sequence_max_length']) for _, seq in records])
    print("[SEQ] Xseq", Xseq.shape)
else:
    Xseq = None

In [ ]:
# =============================
# 5. Predict branch probabilities and ensemble
# =============================
@torch.no_grad()
def predict_numpy(model, X, batch_size):
    model = model.to(DEVICE).eval()
    loader = DataLoader(TensorDataset(torch.from_numpy(X)), batch_size=batch_size, shuffle=False, num_workers=0)
    chunks = []
    for (xb,) in loader:
        xb = xb.to(DEVICE)
        chunks.append(torch.sigmoid(model(xb)).cpu().numpy().astype(np.float32))
    return np.vstack(chunks)

branch_probs = {}

if 'esm_mlp' in model_states:
    model = EmbeddingMLP(Xesm.shape[1], output_dim, cfg['esm_hidden_dims'], cfg['esm_dropout'])
    model.load_state_dict(model_states['esm_mlp'])
    branch_probs['esm_mlp'] = predict_numpy(model, Xesm, int(cfg.get('esm_batch_size', 512)))
    print('[PRED] esm_mlp', branch_probs['esm_mlp'].shape)
    del model
    if DEVICE == 'cuda': torch.cuda.empty_cache()

if 'protcnn' in model_states:
    model = ProtCNN(output_dim, dropout=cfg['protcnn_dropout'])
    model.load_state_dict(model_states['protcnn'])
    branch_probs['protcnn'] = predict_numpy(model, Xseq, int(cfg.get('sequence_batch_size', 256)))
    print('[PRED] protcnn', branch_probs['protcnn'].shape)
    del model
    if DEVICE == 'cuda': torch.cuda.empty_cache()

if 'bilstm_attention' in model_states:
    model = BiLSTMAttention(output_dim, dropout=cfg['bilstm_dropout'])
    model.load_state_dict(model_states['bilstm_attention'])
    bs = max(32, int(cfg.get('sequence_batch_size', 256)) // 4)
    branch_probs['bilstm_attention'] = predict_numpy(model, Xseq, bs)
    print('[PRED] bilstm_attention', branch_probs['bilstm_attention'].shape)
    del model
    if DEVICE == 'cuda': torch.cuda.empty_cache()

weights = cfg.get('active_ensemble_weights') or {k: cfg['ensemble_weights'].get(k, 1.0) for k in branch_probs}
weights = {k: weights.get(k, 0.0) for k in branch_probs}
weight_sum = sum(weights.values())
weights = {k: v / weight_sum for k, v in weights.items()}
ensemble_probs = sum(weights[k] * branch_probs[k] for k in weights).astype(np.float32)
print('[ENSEMBLE] weights', weights)
print('[ENSEMBLE] probs', ensemble_probs.shape)

In [ ]:
# =============================
# 6. Build prediction table
# =============================
def prediction_table(records, probs, model_name, top_k=100, threshold=None):
    rows = []
    threshold = float(cfg.get('best_threshold_micro_f1', 0.0)) if threshold is None else float(threshold)
    for i, (pid, _) in enumerate(records):
        order = np.argsort(-probs[i])[:top_k]
        for j in order:
            score = float(probs[i, j])
            if score < threshold:
                continue
            term = selected_terms[j]
            rows.append({
                'Model': model_name,
                'ProteinID': pid,
                'GO_Term': term,
                'Score': round(score, 6),
                'Aspect': go_meta['term_to_aspect'].get(term),
                'Name': go_meta['term_name'].get(term, ''),
            })
    return pd.DataFrame(rows)

all_tables = [prediction_table(records, ensemble_probs, 'ensemble', TOP_K, THRESHOLD)]
for name, probs in branch_probs.items():
    all_tables.append(prediction_table(records, probs, name, TOP_K, THRESHOLD))

pred_df = pd.concat(all_tables, ignore_index=True) if all_tables else pd.DataFrame()
print(f"[OUTPUT] rows={len(pred_df):,}")
display(pred_df.head(50))

out_csv = OUTPUT_DIR / 'new_sequence_predictions_all_models.csv'
out_tsv = OUTPUT_DIR / 'new_sequence_predictions_ensemble.tsv'
pred_df.to_csv(out_csv, index=False)
pred_df[pred_df['Model'] == 'ensemble'][['ProteinID', 'GO_Term', 'Score']].to_csv(out_tsv, sep='\t', index=False, header=False)
print(f"[SAVE] {out_csv}")
print(f"[SAVE] {out_tsv}")

In [ ]:
# =============================
# 7. Inspect top predictions per protein
# =============================
for pid in pred_df['ProteinID'].drop_duplicates().tolist():
    print('=' * 100)
    print(pid)
    display(
        pred_df[(pred_df['ProteinID'] == pid) & (pred_df['Model'] == 'ensemble')]
        .sort_values('Score', ascending=False)
        .head(30)
    )